> **Disclaimer.** This project is educational and is not investment advice. Backtest and optimizer results are historical and do not predict future results.

# Turtle breakout with ATR pyramiding: does it beat the CDI and the IBOV?

The rules are the owner's TradingView script (`turtle/turtle.pine`) as written in `docs/brief.md` section 7c: buy a close above the highest high of the previous 20 candles, add a unit every 1 ATR above the average price (up to 4 units), leave on a close below the lowest low of the previous 20 candles or on a close more than 1 ATR below the average price. Benchmarks are in `docs/cdi.md` and `docs/ibov.md`. The notebook holds no trading logic: it calls `b3core` and prints tables.

**Before trusting any number here:**
- **There is no capital cap (owner's decision, still an open question).** `capital_brl` only sets the risk budget. With a 3% ATR one unit costs about 17,000 and four units about 67,000 against 25,000 of capital. A CAGR on `capital_brl` therefore hides leverage. Every table shows **peak money in the position divided by capital** next to the return: read the two together.
- **Two fill rules.** `signal_close` is the script (every order fills at the signal candle's close, which is optimistic: the real close is an auction). `next_open` fills at the next open and is the realistic one. If they differ a lot, the strategy depends on the optimistic fill. Section 8 compares them.
- **The stop is read on the close**, not a resting order, so a gap can lose more than one ATR.
- **The fixed setting is the script's own** (20, 20, 14, 1 ATR, 4 units): no value was tuned on this data.
- **The ticker lists are drafts from memory, and "most traded" uses recent data.** That is survivorship bias. Tickers with no data are listed, never silently dropped.
- **BDRs carry the dollar**, and their Yahoo history is short and often flat. The IBOV and the CDI are not like-for-like for them.
- **Not modelled:** taxes, slippage beyond the 0.03% fee on every fill, dividends (raw closes), and a shared-cash rule for simultaneous signals. The portfolio is independent slots (`b3core.portfolio`).
- **The engine counts, and warns about, two things:** units skipped because they rounded below one lot, and candles that justified more than one add (only one is made). The tables below sum them instead of printing a warning per run.

In [1]:
import json
import logging
import urllib.request
import warnings
from dataclasses import replace
from datetime import UTC, date, datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf

from b3core.benchmark import compare_with_index
from b3core.portfolio import buy_and_hold_equity, combine_equity, summarize_portfolio
from b3core.turtle import TurtleConfig, run_turtle
from b3core.universe import scan_universe

pd.options.display.float_format = "{:,.4f}".format
CACHE = Path("../samples/ifr2_universe")  # gitignored vendor data, shared with the IFR2 notebooks
CACHE.mkdir(parents=True, exist_ok=True)

logging.getLogger("yfinance").setLevel(logging.CRITICAL)  # failed downloads are listed below, once


In [2]:
print(f"Run on {date.today():%Y-%m-%d}. Prices are raw Yahoo Finance closes up to the latest session; the CDI source is printed in section 3.")

Run on 2026-10-21. Prices are raw Yahoo Finance closes up to the latest session; the CDI source is printed in section 3.


## 1. Parameters

`CONFIG` is the script's own setting plus the B3 fee of 0.03% on every fill (`TurtleConfig.preset`). Everything else is a list or a size you can change.

In [3]:
CONFIG = TurtleConfig.preset()  # entry 20, exit 20, ATR 14, add every 1 ATR, 4 units, capital 25,000, risk 2%, fee 0.03%
START = date(2012, 1, 1)
END = date(2027, 1, 1)  # exclusive
OOS_FRACTION = 0.3

# Draft candidates, from memory: edit freely. Tickers that fail to download are reported below.
IBOV_CANDIDATES = [
    "PETR4", "VALE3", "ITUB4", "BBDC4", "BBAS3", "ABEV3", "WEGE3", "RENT3", "LREN3", "MGLU3",
    "SUZB3", "GGBR4", "CSNA3", "USIM5", "JBSS3", "RADL3", "RAIL3", "EQTL3", "CMIG4", "ELET3",
    "TOTS3", "PRIO3", "CSAN3", "BBSE3", "SBSP3", "VIVT3", "B3SA3", "BRAP4", "GOAU4", "CYRE3",
    "ITSA4", "KLBN11", "TAEE11", "SANB11", "BPAC11", "HAPV3", "COGN3", "BEEF3", "MRFG3", "CPLE6",
    "EGIE3", "SMTO3", "RDOR3", "HYPE3", "CCRO3", "UGPA3", "VBBR3", "EMBR3", "ALPA4", "MULT3",
    "BRFS3", "ASAI3", "GMAT3", "CMIN3", "ENEV3", "RRRP3",
]
BDR_CANDIDATES = [
    "AAPL34", "MSFT34", "AMZO34", "GOGL34", "NVDC34", "TSLA34", "M1TA34", "NFLX34",
    "DISB34", "COCA34", "JPMC34", "BERK34", "MUTC34", "ROXO34", "MELI34", "BABA34",
]
N_LIQUID_IBOV = 30  # the Ibovespa universe: the most traded of the candidates
N_LIQUID_BDR = 10  # the BDR universe: the ten most traded of the candidates
PICK_SIZES = [5, 8, 10]  # how many Ibovespa names to keep when choosing the best in-sample
MIN_CANDLES_IBOV = 750  # about three years of daily candles
MIN_CANDLES_BDR = 500  # BDRs are younger: about two years
LIQUIDITY_WINDOW = 250  # candles used to measure the median traded value

## 2. Load candles

Daily candles from Yahoo Finance, raw prices, one CSV per ticker in the shared cache folder (the files `ifr2_universe.ipynb` already downloaded are reused). A **data-quality table** follows, because thin history and flat days matter most for BDRs.

In [4]:
def load_candles(ticker: str) -> pd.DataFrame | None:
    path = CACHE / f"{ticker}.csv"
    if not path.exists():
        raw = yf.download(f"{ticker}.SA", start=START, end=END, auto_adjust=False, progress=False)
        if raw.empty:
            return None
        raw.columns = raw.columns.get_level_values(0)
        raw = raw.dropna(subset=["Open", "High", "Low", "Close"])
        raw[["Open", "High", "Low", "Close", "Volume"]].to_csv(path)
    raw = pd.read_csv(path, index_col=0, parse_dates=True)
    return pd.DataFrame(
        {
            "ts": raw.index.tz_localize("America/Sao_Paulo").tz_convert("UTC"),
            "open": raw["Open"].to_numpy(),
            "high": raw["High"].to_numpy(),
            "low": raw["Low"].to_numpy(),
            "close": raw["Close"].to_numpy(),
            "volume": raw["Volume"].to_numpy(dtype=float),
        }
    ).reset_index(drop=True)


def load_ibov() -> pd.Series:
    path = CACHE / "ibov.csv"
    if not path.exists():
        raw = yf.download("^BVSP", start=START, end=END, auto_adjust=False, progress=False)
        raw.columns = raw.columns.get_level_values(0)
        raw[["Close"]].dropna().to_csv(path)
    return pd.read_csv(path, index_col=0, parse_dates=True)["Close"].dropna()


def local_day(candles: pd.DataFrame) -> pd.DatetimeIndex:
    return pd.DatetimeIndex(candles["ts"]).tz_convert("America/Sao_Paulo").tz_localize(None).normalize()


ibov = load_ibov()


def load_universe(tickers: list[str], min_candles: int) -> tuple[dict, dict]:
    """Raw candles per ticker, and the tickers left out (no data, or too little history)."""
    loaded = {t: load_candles(t) for t in tickers}
    out = {
        "no data": [t for t, c in loaded.items() if c is None],
        "too short": [t for t, c in loaded.items() if c is not None and len(c) < min_candles],
    }
    data = {t: c for t, c in loaded.items() if c is not None and len(c) >= min_candles}
    return data, out


ibov_all, ibov_out = load_universe(IBOV_CANDIDATES, MIN_CANDLES_IBOV)
bdr_all, bdr_out = load_universe(BDR_CANDIDATES, MIN_CANDLES_BDR)
print(f"Ibovespa candidates: {len(ibov_all)} loaded; left out: {ibov_out}")
print(f"BDR candidates: {len(bdr_all)} loaded; left out: {bdr_out}")
print(f"IBOV: {len(ibov)} closes, {ibov.index[0].date()} to {ibov.index[-1].date()}")

Ibovespa candidates: 48 loaded; left out: {'no data': ['JBSS3', 'ELET3', 'MRFG3', 'CPLE6', 'CCRO3', 'EMBR3', 'BRFS3', 'RRRP3'], 'too short': []}
BDR candidates: 16 loaded; left out: {'no data': [], 'too short': []}
IBOV: 3667 closes, 2012-01-03 to 2026-10-16


In [5]:
def quality(data: dict) -> pd.DataFrame:
    """Plain data checks: history, zero-volume days, flat days (high equals low), the longest gap."""
    rows = {}
    for t, c in data.items():
        days = local_day(c)
        rows[t] = {
            "candles": len(c), "first": days[0].date(), "last": days[-1].date(),
            "zero volume %": float((c["volume"] == 0).mean() * 100),
            "flat days %": float((c["high"] == c["low"]).mean() * 100),
            "longest gap (days)": int(pd.Series(days).diff().dt.days.max()),
        }
    return pd.DataFrame(rows).T


quality(bdr_all)

,candles,first,last,zero volume %,flat days %,longest gap (days)
AAPL34,3645,2012-02-23,2026-10-21,13.0041,19.8354,5
MSFT34,3679,2012-01-02,2026-10-21,17.6407,32.4273,6
AMZO34,3658,2012-02-01,2026-10-21,29.5243,44.2865,6
GOGL34,2607,2016-05-09,2026-10-21,2.1097,7.1346,5
NVDC34,1855,2019-05-13,2026-10-21,5.9299,9.7035,5
TSLA34,2430,2017-01-19,2026-10-21,22.8395,27.9012,5
M1TA34,3141,2014-03-10,2026-10-21,21.6492,33.4607,5
NFLX34,3554,2012-07-04,2026-10-21,37.9010,49.9719,5
DISB34,3660,2012-01-30,2026-10-21,21.7486,36.2022,6
COCA34,3679,2012-01-02,2026-10-21,35.8793,50.6931,6


## 3. The CDI

The benchmark is the daily CDI compounded over each window (`docs/cdi.md`). The Central Bank API is tried first; if it is blocked, the **Selic target minus 0.10 points a year** is used, which is an approximation (the cell prints which one). A file `samples/ifr2_universe/cdi.csv` (series 12, columns `date` and `rate`) overrides both.

In [6]:
def fetch(url: str) -> str:
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=60) as response:
        return response.read().decode("utf-8")


trading_days = sorted(set().union(*[set(local_day(c)) for d in (ibov_all, bdr_all) for c in d.values()]))


def cdi_from_bcb() -> pd.Series:
    rows, begin = [], START
    while begin < END and begin < datetime.now(UTC).date():
        stop = min(begin + timedelta(days=3_000), END - timedelta(days=1))  # about 8 years
        url = (
            "https://api.bcb.gov.br/dados/serie/bcdata.sgs.12/dados"
            f"?formato=json&dataInicial={begin:%d/%m/%Y}&dataFinal={stop:%d/%m/%Y}"
        )
        rows += json.loads(fetch(url))  # raises when the API answers with an HTML error page
        begin = stop + timedelta(days=1)
    index = pd.to_datetime([r["data"] for r in rows], format="%d/%m/%Y")
    return pd.Series([float(r["valor"]) for r in rows], index=index).sort_index()


def cdi_from_selic_target() -> pd.Series:
    url = "http://www.ipeadata.gov.br/api/odata4/ValoresSerie(SERCODIGO='BM366_TJOVER366')"
    values = json.loads(fetch(url))["value"]
    annual = pd.Series(
        [v["VALVALOR"] for v in values], index=pd.to_datetime([v["VALDATA"][:10] for v in values])
    )
    daily = (1 + (annual - 0.10) / 100) ** (1 / 252) - 1  # percent a year to a business-day rate
    return (daily * 100).reindex(trading_days).dropna()  # trading days only


def load_cdi() -> tuple[pd.Series, str]:
    exact = CACHE / "cdi.csv"
    approx = CACHE / "cdi_from_selic_target.csv"
    if not exact.exists() and not approx.exists():
        try:
            cdi_from_bcb().rename_axis("date").rename("rate").to_csv(exact, date_format="%d/%m/%Y")
        except Exception as error:  # the API blocks some networks
            print(f"Central Bank API unavailable ({type(error).__name__}); using the Selic target")
            cdi_from_selic_target().rename_axis("date").rename("rate").to_csv(approx, date_format="%d/%m/%Y")
    path = exact if exact.exists() else approx
    frame = pd.read_csv(path)
    index = pd.to_datetime(frame["date"], format="%d/%m/%Y")
    series = pd.Series(frame["rate"].to_numpy(dtype=float), index=index).sort_index()
    label = "CDI (BCB series 12)" if path == exact else "Selic target minus 0.10 points (approximation)"
    return series, label


cdi, CDI_SOURCE = load_cdi()
print(f"source: {CDI_SOURCE}")
print(f"{len(cdi)} days, {cdi.index[0].date()} to {cdi.index[-1].date()}")

source: CDI (BCB series 12)
3715 days, 2012-01-02 to 2026-10-16


## 4. Clean the candles, then build the two universes

Two kinds of row cannot be compared with the CDI, and they are **dropped and printed**, never filled:
- **Candles before the first IBOV close** (`docs/ibov.md`): Yahoo's index series starts a day after the first stock candle.
- **Candles on days with no CDI rate.** With the exact CDI (series 12) these are holidays that Yahoo pads with a zero-volume row, and the days after the latest published rate. With the Selic-target fallback every weekday has a rate, so this check drops nothing.

**Most traded** is then the median of `close * volume` over the last `LIQUIDITY_WINDOW` candles of each ticker. It is measured on the most recent data, so it is a survivor ranking, not what was liquid in 2013.

In [7]:
def clean(data: dict) -> tuple[dict, pd.DataFrame]:
    cleaned, rows = {}, {}
    cdi_days = set(pd.DatetimeIndex(cdi.index).normalize())
    for t, c in data.items():
        days = local_day(c)
        before_ibov = days < ibov.index[0]
        no_rate = ~before_ibov & ~days.isin(cdi_days) & (days > ibov.index[0])
        zero_volume = no_rate & (c["volume"].to_numpy() == 0)
        keep = ~before_ibov & ~no_rate
        cleaned[t] = c[keep].reset_index(drop=True)
        rows[t] = {
            "before first IBOV close": int(before_ibov.sum()),
            "no CDI rate": int(no_rate.sum()),
            "...of which zero volume": int(zero_volume.sum()),
        }
    return cleaned, pd.DataFrame(rows).T


def traded_value(data: dict) -> pd.Series:
    return pd.Series(
        {t: float((c["close"] * c["volume"]).tail(LIQUIDITY_WINDOW).median()) for t, c in data.items()}
    ).sort_values(ascending=False)


ibov_clean, ibov_dropped = clean(ibov_all)
bdr_clean, bdr_dropped = clean(bdr_all)
dropped = pd.concat([ibov_dropped, bdr_dropped])
print(f"{int((dropped > 0).any(axis=1).sum())} of {len(dropped)} tickers had rows dropped. Most dropped for one ticker:")
print(dropped.max().to_string())

ibov_liquidity = traded_value(ibov_clean)
bdr_liquidity = traded_value(bdr_clean)
ibov_data = {t: ibov_clean[t] for t in ibov_liquidity.index[:N_LIQUID_IBOV]}
bdr_data = {t: bdr_clean[t] for t in bdr_liquidity.index[:N_LIQUID_BDR]}
print(f"Ibovespa universe ({len(ibov_data)}): {sorted(ibov_data)}")
print(f"BDR universe ({len(bdr_data)}): {sorted(bdr_data)}")
pd.DataFrame({"ibov: median traded value (R$)": ibov_liquidity, "bdr: median traded value (R$)": bdr_liquidity})

64 of 64 tickers had rows dropped. Most dropped for one ticker:
before first IBOV close    1
no CDI rate                9
...of which zero volume    6
Ibovespa universe (30): ['ABEV3', 'B3SA3', 'BBAS3', 'BBDC4', 'BBSE3', 'BPAC11', 'CMIG4', 'CSAN3', 'CYRE3', 'ENEV3', 'EQTL3', 'GGBR4', 'ITSA4', 'ITUB4', 'LREN3', 'MGLU3', 'PETR4', 'PRIO3', 'RADL3', 'RAIL3', 'RDOR3', 'RENT3', 'SBSP3', 'SUZB3', 'TOTS3', 'UGPA3', 'VALE3', 'VBBR3', 'VIVT3', 'WEGE3']
BDR universe (10): ['AAPL34', 'AMZO34', 'GOGL34', 'M1TA34', 'MELI34', 'MSFT34', 'MUTC34', 'NVDC34', 'ROXO34', 'TSLA34']


,ibov: median traded value (R$),bdr: median traded value (R$)
AAPL34,NaN,"19,695,111.9603"
ABEV3,"384,428,898.6681",NaN
ALPA4,"22,380,945.2724",NaN
AMZO34,NaN,"38,041,225.4244"
ASAI3,"114,016,927.9548",NaN
...,...,...
USIM5,"78,229,522.2545",NaN
VALE3,"1,500,479,277.6146",NaN
VBBR3,"267,888,262.6926",NaN
VIVT3,"162,688,094.2786",NaN


## 5. One shared out-of-sample date per universe

Each universe is cut at the date that leaves roughly the last `OOS_FRACTION` of its trading days out. Everything that is *chosen* (which Ibovespa names to keep) is chosen on the in-sample part only.

In [8]:
def split(data: dict) -> tuple[dict, dict, pd.Timestamp]:
    days = {t: local_day(c) for t, c in data.items()}
    all_days = sorted(set().union(*days.values()))
    cut = all_days[int(len(all_days) * (1 - OOS_FRACTION))]
    return (
        {t: c[days[t] < cut].reset_index(drop=True) for t, c in data.items()},
        {t: c[days[t] >= cut].reset_index(drop=True) for t, c in data.items()},
        cut,
    )


ibov_is, ibov_oos, IBOV_CUT = split(ibov_data)
bdr_is, bdr_oos, BDR_CUT = split(bdr_data)
print(f"Ibovespa universe: out-of-sample starts {IBOV_CUT.date()}")
print(f"BDR universe: out-of-sample starts {BDR_CUT.date()}")

Ibovespa universe: out-of-sample starts 2022-05-23
BDR universe: out-of-sample starts 2022-05-23


## 6. Helpers: running Turtle over a universe

`run_all` runs the fixed setting on every ticker. The engine's warnings (skipped units, several adds due on one candle) would print once per run, so they are silenced here and **summed in the tables** instead. `exposure` shows what the strategy actually did: how many units, how much money, how often it was in a trade.

In [9]:
def run_all(data: dict, config: TurtleConfig) -> dict:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")  # the counts are in the result and summed in the tables
        return {t: run_turtle(c, config) for t, c in data.items()}


def exposure(data: dict, config: TurtleConfig) -> pd.DataFrame:
    """What the strategy did on each ticker: trades, units, money in the position, time in trades."""
    rows = {}
    for t, r in run_all(data, config).items():
        trades = r.trades
        rows[t] = {
            "trades": len(trades),
            "avg units": float(np.mean([x.units for x in trades])) if trades else 0.0,
            "max units": max((x.units for x in trades), default=0),
            "% candles in a closed trade": 100 * sum(x.days for x in trades) / len(data[t]),
            "peak money / capital": r.peak_cost / config.capital_brl,
            "skipped units": r.skipped_units,
            "multi-add candles": r.multi_add_candles,
            "open units at end": r.open_units,
        }
    return pd.DataFrame(rows).T


SCAN_COLUMNS = ["n_trades", "result", "cagr", "excess", "beats_cdi", "max_equity_drawdown", "calmar", "score_v1",
                "bh_cagr", "beats_bh", "ibov_cagr", "beats_ibov"]


def scan_one(data: dict, config: TurtleConfig, min_candles: int) -> pd.DataFrame:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return scan_universe(data, config, cdi, min_candles=min_candles, index_closes=ibov, runner=run_turtle)


def scan_both(in_sample: dict, out_of_sample: dict, config: TurtleConfig, min_is: int, min_oos: int) -> pd.DataFrame:
    scan_is = scan_one(in_sample, config, min_is)
    scan_oos = scan_one(out_of_sample, config, min_oos)
    print("skipped in-sample:", scan_is.attrs["skipped"] or "none", "| out-of-sample:", scan_oos.attrs["skipped"] or "none")
    table = scan_is[SCAN_COLUMNS].add_prefix("is_").join(scan_oos[SCAN_COLUMNS].add_prefix("oos_"), how="left")
    return table.sort_values("is_score_v1", ascending=False)

## 7. Ibovespa names: the script's setting on every ticker

The fixed `CONFIG` on each ticker, in-sample and out-of-sample side by side. `excess` is the strategy's CAGR minus the CDI's over the same window, `beats_cdi` is true when it is positive. Read `max_equity_drawdown` and `calmar` next to it, and the exposure table below for the leverage hidden behind the CAGR.

In [10]:
scan_ibov = scan_both(ibov_is, ibov_oos, CONFIG, MIN_CANDLES_IBOV // 2, 120)
scan_ibov

skipped in-sample: ['RDOR3'] | out-of-sample: none


,is_n_trades,is_result,is_cagr,is_excess,is_beats_cdi,is_max_equity_drawdown,is_calmar,is_score_v1,is_bh_cagr,is_beats_bh,...,oos_cagr,oos_excess,oos_beats_cdi,oos_max_equity_drawdown,oos_calmar,oos_score_v1,oos_bh_cagr,oos_beats_bh,oos_ibov_cagr,oos_beats_ibov
RENT3,38,3.2952,0.1508,0.0665,True,0.4842,0.3114,36.3235,0.2052,False,...,-0.1175,-0.2487,False,0.5261,-0.2234,0.0000,-0.0262,False,0.1517,False
WEGE3,38,3.7952,0.1631,0.0788,True,0.3988,0.4089,36.3235,0.2384,False,...,0.1768,0.0457,True,0.5684,0.3111,19.7800,0.1438,True,0.1517,True
RADL3,37,3.6921,0.1606,0.0763,True,0.3882,0.4139,35.8955,0.2034,False,...,-0.1086,-0.2398,False,0.5717,-0.1900,0.0000,0.0360,False,0.1517,False
LREN3,34,3.1283,0.1464,0.0621,True,0.2918,0.5017,35.0373,0.1248,True,...,0.0053,-0.1258,False,0.3851,0.0139,3.1156,-0.1148,True,0.1517,False
PRIO3,33,8.6315,0.2439,0.1596,True,0.5000,0.4878,34.0476,0.0933,True,...,0.1660,0.0349,True,0.4940,0.3361,24.5637,0.2510,False,0.1517,True
ENEV3,32,2.8970,0.1401,0.0557,True,1.5555,0.0900,33.5484,-0.2145,True,...,0.1385,0.0073,True,0.3860,0.3587,20.6818,0.1934,False,0.1517,False
EQTL3,32,3.1615,0.1473,0.0630,True,0.5013,0.2938,33.5484,0.2426,False,...,-0.1551,-0.2862,False,0.7169,-0.2164,0.0000,0.1807,False,0.1517,False
MGLU3,28,9.3034,0.2521,0.1677,True,0.4404,0.5723,31.3793,0.2776,False,...,-0.6231,-0.7543,False,0.9915,-0.6285,0.0000,-0.2190,False,0.1517,False
CYRE3,32,1.4958,0.0921,0.0078,True,0.6181,0.1491,30.9076,-0.0084,True,...,0.1541,0.0229,True,0.4641,0.3320,18.3170,0.1945,False,0.1517,True
GGBR4,47,1.3766,0.0870,0.0027,True,0.4829,0.1802,28.6726,0.0608,True,...,-0.0801,-0.2112,False,0.7688,-0.1041,0.0000,0.0247,False,0.1517,False


In [11]:
both = scan_ibov.dropna(subset=["oos_beats_cdi"])
pd.Series(
    {
        "tickers": len(scan_ibov),
        "beat the CDI in-sample": int(scan_ibov["is_beats_cdi"].sum()),
        "beat the CDI out-of-sample": int(both["oos_beats_cdi"].sum()),
        "beat it in both": int((both["is_beats_cdi"] & both["oos_beats_cdi"]).sum()),
        "beat the IBOV out-of-sample": int(both["oos_beats_ibov"].sum()),
        "median in-sample excess": scan_ibov["is_excess"].median(),
        "median out-of-sample excess": both["oos_excess"].median(),
    },
    name="summary",
).to_frame()

,summary
tickers,29.0000
beat the CDI in-sample,12.0000
beat the CDI out-of-sample,8.0000
beat it in both,5.0000
beat the IBOV out-of-sample,6.0000
median in-sample excess,-0.0106
median out-of-sample excess,-0.1306


**What the strategy did, full history.** `peak money / capital` is the most money the open units ever held divided by `capital_brl`: 1 means the whole capital, 3 means three times it (leverage the spec does not allow yet). `% candles in a closed trade` is the share of candles inside a position that has closed.

In [12]:
exposure(ibov_data, CONFIG).sort_values("peak money / capital", ascending=False)

,trades,avg units,max units,% candles in a closed trade,peak money / capital,skipped units,multi-add candles,open units at end
BBSE3,51.0000,2.6275,4.0000,36.0729,6.8119,0.0000,15.0000,0.0000
VALE3,68.0000,2.2941,4.0000,32.4339,5.5622,0.0000,16.0000,0.0000
ITSA4,56.0000,2.6250,4.0000,41.9460,5.3002,0.0000,14.0000,4.0000
ABEV3,68.0000,1.9853,4.0000,32.2976,5.0921,0.0000,18.0000,0.0000
ENEV3,47.0000,2.0000,4.0000,29.1905,4.8853,0.0000,8.0000,4.0000
CMIG4,63.0000,2.2698,4.0000,33.7694,4.8339,0.0000,12.0000,1.0000
VIVT3,73.0000,2.0411,4.0000,28.8362,4.8243,0.0000,16.0000,0.0000
BBAS3,70.0000,2.3571,4.0000,39.4113,4.7470,0.0000,21.0000,0.0000
ITUB4,62.0000,2.3871,4.0000,38.7299,4.7213,0.0000,18.0000,4.0000
WEGE3,56.0000,2.6071,4.0000,45.0531,4.7006,0.0000,19.0000,0.0000


## 8. The two fill rules, side by side

`signal_close` is the script. `next_open` signals on the close and fills at the next open, with the quantity and the levels of the signal candle. The same setting, the same tickers, the same windows. `beat` columns count tickers. A big gap between the rows means the result depends on the optimistic close fill.

In [13]:
def compare_settings(settings: dict, in_sample: dict, out_of_sample: dict, min_is: int, min_oos: int) -> pd.DataFrame:
    rows = []
    for label, config in settings.items():
        for window, data, min_candles in (("in-sample", in_sample, min_is), ("out-of-sample", out_of_sample, min_oos)):
            table = scan_one(data, config, min_candles)
            results = {t: r for t, r in run_all(data, config).items() if t in table.index}
            peaks = pd.Series({t: r.peak_cost / config.capital_brl for t, r in results.items()})
            rows.append({
                "setting": label, "window": window, "tickers": len(table),
                "median cagr": table["cagr"].median(), "median excess": table["excess"].median(),
                "beat CDI": int(table["beats_cdi"].sum()), "beat B&H": int(table["beats_bh"].sum()),
                "beat IBOV": int(table["beats_ibov"].sum()),
                "median equity dd": table["max_equity_drawdown"].median(),
                "median peak money / capital": peaks.median(), "max peak money / capital": peaks.max(),
                "skipped units": sum(r.skipped_units for r in results.values()),
                "multi-add candles": sum(r.multi_add_candles for r in results.values()),
            })
    return pd.DataFrame(rows).set_index(["setting", "window"])


FILLS = {
    "signal_close (script)": CONFIG,
    "next_open": replace(CONFIG, fill_at="next_open"),
}
fill_study = compare_settings(FILLS, ibov_is, ibov_oos, MIN_CANDLES_IBOV // 2, 120)
fill_study

tickers  median cagr  median excess  \
setting               window                                               
signal_close (script) in-sample           29       0.0737        -0.0106   
                      out-of-sample       30       0.0029        -0.1282   
next_open             in-sample           29       0.0704        -0.0139   
                      out-of-sample       30      -0.0166        -0.1477   

                                     beat CDI  beat B&H  beat IBOV  \
setting               window                                         
signal_close (script) in-sample            12        10         16   
                      out-of-sample         8         4          6   
next_open             in-sample            10        10         17   
                      out-of-sample         7         7          4   

                                     median equity dd  \
setting               window                            
signal_close (script) in-sample                0.5432   
                      out-of-sample            0.5365   
next_open             in-sample                0.5523   
                      out-of-sample            0.5769   

                                     median peak money / capital  \
setting               window                                       
signal_close (script) in-sample                           3.9815   
                      out-of-sample                       3.6840   
next_open             in-sample                           3.9800   
                      out-of-sample                       3.6053   

                                     max peak money / capital  skipped units  \
setting               window                                                   
signal_close (script) in-sample                        5.0921              0   
                      out-of-sample                    6.8119              0   
next_open             in-sample                        5.2825              0   
                      out-of-sample                    6.8129              0   

                                     multi-add candles  
setting               window                            
signal_close (script) in-sample                    238  
                      out-of-sample                156  
next_open             in-sample                    241  
                      out-of-sample                133

## 9. Does pyramiding help?

`max_units` 1 is the same breakout and exit with no adds, so the difference to 4 is what the pyramiding adds (return, drawdown and money in the position). Same fill rule as the script.

In [14]:
UNITS = {f"max_units {n}": replace(CONFIG, max_units=n) for n in (1, 2, 3, 4)}
units_study = compare_settings(UNITS, ibov_is, ibov_oos, MIN_CANDLES_IBOV // 2, 120)
units_study

tickers  median cagr  median excess  beat CDI  \
setting     window                                                         
max_units 1 in-sample           29       0.0326        -0.0517         4   
            out-of-sample       30       0.0217        -0.1095         0   
max_units 2 in-sample           29       0.0602        -0.0241         9   
            out-of-sample       30       0.0106        -0.1206         1   
max_units 3 in-sample           29       0.0601        -0.0242        11   
            out-of-sample       30       0.0091        -0.1220         4   
max_units 4 in-sample           29       0.0737        -0.0106        12   
            out-of-sample       30       0.0029        -0.1282         8   

                           beat B&H  beat IBOV  median equity dd  \
setting     window                                                 
max_units 1 in-sample             5          6            0.3211   
            out-of-sample         6          0            0.2308   
max_units 2 in-sample             8         15            0.4219   
            out-of-sample         6          1            0.3587   
max_units 3 in-sample             8         15            0.4930   
            out-of-sample         4          2            0.4761   
max_units 4 in-sample            10         16            0.5432   
            out-of-sample         4          6            0.5365   

                           median peak money / capital  \
setting     window                                       
max_units 1 in-sample                           0.9865   
            out-of-sample                       0.9633   
max_units 2 in-sample                           1.9642   
            out-of-sample                       1.8038   
max_units 3 in-sample                           3.0114   
            out-of-sample                       2.7384   
max_units 4 in-sample                           3.9815   
            out-of-sample                       3.6840   

                           max peak money / capital  skipped units  \
setting     window                                                   
max_units 1 in-sample                        3.7872              0   
            out-of-sample                    1.6831              0   
max_units 2 in-sample                        3.7872              0   
            out-of-sample                    3.4727              0   
max_units 3 in-sample                        3.7872              0   
            out-of-sample                    5.1180              0   
max_units 4 in-sample                        5.0921              0   
            out-of-sample                    6.8119              0   

                           multi-add candles  
setting     window                            
max_units 1 in-sample                      0  
            out-of-sample                  0  
max_units 2 in-sample                      0  
            out-of-sample                  0  
max_units 3 in-sample                     77  
            out-of-sample                 54  
max_units 4 in-sample                    238  
            out-of-sample                156

## 10. A portfolio of independent slots

One ticker leaves most of the capital idle, so the capital is split equally between the tickers (`b3core.portfolio`, independent slots of `capital / N`, no shared-cash rule). Each slot keeps `risk_pct` of **its own** capital, so the portfolio's risk per unit shrinks as N grows, and there is still no cap on the money in a slot. `sum of slot peaks / capital` adds the peak of every slot: an upper bound for the money the portfolio could hold at once, because the peaks need not coincide.

In [15]:
CAPITAL = CONFIG.capital_brl


def slot_curves(data: dict, n: int, config: TurtleConfig) -> tuple[dict, dict]:
    """The equity curve and the run of each ticker in a slot of CAPITAL / n."""
    slot = replace(config, capital_brl=CAPITAL / n)
    runs = run_all(data, slot)
    curves = {t: pd.Series(r.equity, index=pd.DatetimeIndex(data[t]["ts"])) for t, r in runs.items()}
    return curves, runs


def portfolio_of(curves: dict, runs: dict, members) -> dict:
    equity = combine_equity({t: curves[t] for t in members})
    summary = summarize_portfolio(equity, CAPITAL, cdi)
    held_index = compare_with_index(summary.cagr, summary.years, ibov, equity.index)
    return {
        **summary.__dict__, "ibov_cagr": held_index.cagr, "beats_ibov": held_index.beats,
        "sum of slot peaks / capital": sum(runs[t].peak_cost for t in members) / CAPITAL,
    }


def held_of(data: dict, members) -> dict:
    held = summarize_portfolio(buy_and_hold_equity({t: data[t] for t in members}, CAPITAL), CAPITAL, cdi)
    return {"bh_cagr": held.cagr, "bh_max_equity_drawdown": held.max_equity_drawdown}


def all_tickers_portfolio(universe: str, in_sample: dict, out_of_sample: dict) -> pd.DataFrame:
    rows = []
    for fill, config in FILLS.items():
        for window, data in (("in-sample", in_sample), ("out-of-sample", out_of_sample)):
            names = sorted(data)
            curves, runs = slot_curves(data, len(names), config)
            result = portfolio_of(curves, runs, names)
            held = held_of(data, names)
            rows.append({
                "universe": universe, "fill": fill, "window": window, "tickers": len(names),
                "cagr": result["cagr"], "equity dd": result["max_equity_drawdown"], "calmar": result["calmar"],
                "excess vs CDI": result["excess"], "beats CDI": result["beats_cdi"],
                "IBOV cagr": result["ibov_cagr"], "beats IBOV": result["beats_ibov"],
                "B&H cagr": held["bh_cagr"], "beats B&H": result["cagr"] > held["bh_cagr"],
                "sum of slot peaks / capital": result["sum of slot peaks / capital"],
            })
    return pd.DataFrame(rows).set_index(["universe", "fill", "window"])


all_tickers_portfolio("Ibovespa", ibov_is, ibov_oos)

tickers   cagr  equity dd  \
universe fill                  window                                     
Ibovespa signal_close (script) in-sample           30 0.0991     0.1843   
                               out-of-sample       30 0.0770     0.1874   
         next_open             in-sample           30 0.1003     0.1628   
                               out-of-sample       30 0.0701     0.1900   

                                              calmar  excess vs CDI  \
universe fill                  window                                 
Ibovespa signal_close (script) in-sample      0.5375         0.0147   
                               out-of-sample  0.4109        -0.0541   
         next_open             in-sample      0.6160         0.0160   
                               out-of-sample  0.3689        -0.0611   

                                              beats CDI  IBOV cagr  \
universe fill                  window                                
Ibovespa signal_close (script) in-sample           True     0.0600   
                               out-of-sample      False     0.1517   
         next_open             in-sample           True     0.0600   
                               out-of-sample      False     0.1517   

                                              beats IBOV  B&H cagr  beats B&H  \
universe fill                  window                                           
Ibovespa signal_close (script) in-sample            True    0.1148      False   
                               out-of-sample       False    0.1101      False   
         next_open             in-sample            True    0.1148      False   
                               out-of-sample       False    0.1101      False   

                                              sum of slot peaks / capital  
universe fill                  window                                      
Ibovespa signal_close (script) in-sample                           3.7628  
                               out-of-sample                       3.7802  
         next_open             in-sample                           3.7836  
                               out-of-sample                       3.8383

### 10b. Keep the best N in-sample, judge them out-of-sample

The tickers are ranked by their in-sample `score_v1`, the best N form an equal-split portfolio, and it is judged on the out-of-sample part. The **worst N** are the control. `beats % of random` is the share of random N-ticker portfolios from the same universe with a lower out-of-sample CAGR: near 50% means the choice is no better than chance. Both fill rules.

In [16]:
DRAWS = 300  # random subsets per size
rng = np.random.default_rng(0)  # seeded: the same subsets on every run
names = sorted(set(ibov_is) & set(ibov_oos))
by_score = [t for t in scan_ibov["is_score_v1"].dropna().sort_values(ascending=False).index if t in names]

rows = []
for n in PICK_SIZES:
    random_sets = [tuple(rng.choice(names, size=n, replace=False)) for _ in range(DRAWS)]
    for fill, config in FILLS.items():
        curves, runs = slot_curves({t: ibov_oos[t] for t in names}, n, config)
        random_cagr = pd.Series([portfolio_of(curves, runs, s)["cagr"] for s in random_sets])
        for label, members in (("best in-sample", by_score[:n]), ("worst in-sample (control)", by_score[-n:])):
            result = portfolio_of(curves, runs, members)
            held = held_of(ibov_oos, members)
            rows.append({
                "selection": label, "tickers": n, "fill": fill,
                "oos cagr": result["cagr"], "oos equity dd": result["max_equity_drawdown"],
                "oos calmar": result["calmar"], "oos excess": result["excess"], "beats CDI": result["beats_cdi"],
                "IBOV cagr": result["ibov_cagr"], "beats IBOV": result["beats_ibov"],
                "B&H cagr": held["bh_cagr"], "beats B&H": result["cagr"] > held["bh_cagr"],
                "sum of slot peaks / capital": result["sum of slot peaks / capital"],
                "beats % of random": float((random_cagr < result["cagr"]).mean()),
            })
print("best", PICK_SIZES[0], "in-sample:", by_score[: PICK_SIZES[0]])
pd.DataFrame(rows).set_index(["selection", "tickers", "fill"]).sort_index()

best 5 in-sample: ['RENT3', 'WEGE3', 'RADL3', 'LREN3', 'PRIO3']


oos cagr  \
selection                 tickers fill                              
best in-sample            5       next_open                0.0442   
                                  signal_close (script)    0.0552   
                          8       next_open                0.0337   
                                  signal_close (script)    0.0534   
                          10      next_open                0.0534   
                                  signal_close (script)    0.0663   
worst in-sample (control) 5       next_open                0.0838   
                                  signal_close (script)    0.0768   
                          8       next_open                0.0458   
                                  signal_close (script)    0.0382   
                          10      next_open                0.1074   
                                  signal_close (script)    0.1056   

                                                         oos equity dd  \
selection                 tickers fill                                   
best in-sample            5       next_open                     0.3000   
                                  signal_close (script)         0.2689   
                          8       next_open                     0.3408   
                                  signal_close (script)         0.2885   
                          10      next_open                     0.2875   
                                  signal_close (script)         0.2604   
worst in-sample (control) 5       next_open                     0.3264   
                                  signal_close (script)         0.3695   
                          8       next_open                     0.2584   
                                  signal_close (script)         0.2643   
                          10      next_open                     0.1864   
                                  signal_close (script)         0.1878   

                                                         oos calmar  \
selection                 tickers fill                                
best in-sample            5       next_open                  0.1473   
                                  signal_close (script)      0.2052   
                          8       next_open                  0.0988   
                                  signal_close (script)      0.1850   
                          10      next_open                  0.1858   
                                  signal_close (script)      0.2547   
worst in-sample (control) 5       next_open                  0.2569   
                                  signal_close (script)      0.2079   
                          8       next_open                  0.1773   
                                  signal_close (script)      0.1444   
                          10      next_open                  0.5764   
                                  signal_close (script)      0.5625   

                                                         oos excess  \
selection                 tickers fill                                
best in-sample            5       next_open                 -0.0870   
                                  signal_close (script)     -0.0760   
                          8       next_open                 -0.0975   
                                  signal_close (script)     -0.0778   
                          10      next_open                 -0.0777   
                                  signal_close (script)     -0.0648   
worst in-sample (control) 5       next_open                 -0.0473   
                                  signal_close (script)     -0.0543   
                          8       next_open                 -0.0853   
                                  signal_close (script)     -0.0930   
                          10      next_open                 -0.0237   
                                  signal_close (script)     -0.0255   

                                                         beats 

**How to read it.** A ranking that means something puts `best` above `worst` and above 50% of random. Then ask whether the best row beats the CDI *and* the IBOV with a drawdown you could live with, and whether `sum of slot peaks / capital` says the money needed is a multiple of the capital.

## 11. The ten most traded BDRs

No selection: the universe is already the ten most traded candidates. The same fixed setting on each BDR, then the ten as one equal-split portfolio, both fill rules.

In [17]:
scan_bdr = scan_both(bdr_is, bdr_oos, CONFIG, MIN_CANDLES_BDR // 2, 100)
scan_bdr

skipped in-sample: ['ROXO34'] | out-of-sample: none


,is_n_trades,is_result,is_cagr,is_excess,is_beats_cdi,is_max_equity_drawdown,is_calmar,is_score_v1,is_bh_cagr,is_beats_bh,...,oos_cagr,oos_excess,oos_beats_cdi,oos_max_equity_drawdown,oos_calmar,oos_score_v1,oos_bh_cagr,oos_beats_bh,oos_ibov_cagr,oos_beats_ibov
MSFT34,74,7.8823,0.2343,0.1500,True,0.9250,0.2533,46.2500,0.3668,False,...,-0.0116,-0.1428,False,0.4935,-0.0235,0.0000,0.1873,False,0.1517,False
AMZO34,64,9.7911,0.2599,0.1758,True,0.8726,0.2978,44.2553,0.4076,False,...,0.1139,-0.0172,False,0.6416,0.1776,18.3787,0.2293,False,0.1517,False
M1TA34,51,5.0857,0.2466,0.1620,True,0.4409,0.5592,40.6305,0.2325,True,...,0.3477,0.2165,True,0.2541,1.3681,23.4511,0.3583,False,0.1517,True
GOGL34,32,3.1003,0.2637,0.1933,True,0.4123,0.6396,33.5484,0.2629,True,...,0.3660,0.2349,True,0.4873,0.7511,20.6818,0.3081,True,0.1517,True
AAPL34,63,1.7546,0.1040,0.0200,True,1.6900,0.0616,27.7944,0.3470,False,...,0.1371,0.0060,True,0.3828,0.3582,20.0895,0.2351,False,0.1517,False
TSLA34,22,5.6606,0.4272,0.3658,True,1.6041,0.2663,27.5000,0.7611,False,...,0.0341,-0.0970,False,0.4985,0.0684,14.3128,0.1221,False,0.1517,False
NVDC34,15,4.8878,0.7987,0.7493,True,0.5556,1.4376,21.6667,0.7044,True,...,0.3807,0.2495,True,0.2612,1.4572,29.4134,0.8072,False,0.1517,True
MUTC34,13,3.7136,0.7002,0.6514,True,1.3784,0.5080,19.6512,0.3672,True,...,0.4512,0.3200,True,0.9066,0.4976,28.8889,0.8054,False,0.1517,True
MELI34,10,1.2972,0.3272,0.2784,True,0.7123,0.4594,16.2500,0.1595,True,...,0.0431,-0.0881,False,0.4313,0.0999,13.3971,0.2331,False,0.1517,False


In [18]:
all_tickers_portfolio("BDR", bdr_is, bdr_oos)

tickers   cagr  equity dd  \
universe fill                  window                                     
BDR      signal_close (script) in-sample           10 0.1746     0.2854   
                               out-of-sample       10 0.2353     0.1732   
         next_open             in-sample           10 0.1856     0.3125   
                               out-of-sample       10 0.2285     0.1548   

                                              calmar  excess vs CDI  \
universe fill                  window                                 
BDR      signal_close (script) in-sample      0.6116         0.0902   
                               out-of-sample  1.3585         0.1042   
         next_open             in-sample      0.5938         0.1013   
                               out-of-sample  1.4761         0.0974   

                                              beats CDI  IBOV cagr  \
universe fill                  window                                
BDR      signal_close (script) in-sample           True     0.0600   
                               out-of-sample       True     0.1517   
         next_open             in-sample           True     0.0600   
                               out-of-sample       True     0.1517   

                                              beats IBOV  B&H cagr  beats B&H  \
universe fill                  window                                           
BDR      signal_close (script) in-sample            True    0.2705      False   
                               out-of-sample        True    0.4383      False   
         next_open             in-sample            True    0.2705      False   
                               out-of-sample        True    0.4383      False   

                                              sum of slot peaks / capital  
universe fill                  window                                      
BDR      signal_close (script) in-sample                          10.7842  
                               out-of-sample                       3.3847  
         next_open             in-sample                          10.2147  
                               out-of-sample                       3.3909

**Caveats for the BDRs.** Ten correlated US mega-caps, a short and dollar-driven history, and an out-of-sample window of a few years: this is an anecdote, not evidence. The IBOV comparison is not like-for-like, and there is no US benchmark here.

## 12. What this notebook does not answer (yet)

- **A realistic account.** The capital cap is a deferred rule (`docs/brief.md` section 10). Until it exists, a Turtle CAGR on `capital_brl` should be read together with the money in the position, and a result that needs several times the capital is not a result an account of that size could obtain.
- **Is the part-time exposure judged fairly?** The CDI and the IBOV are fully invested every day, the strategy is not. A comparison over only the days in a trade, or a rule for what idle cash earns, needs an approved spec in `docs/` before any code.
- **Which signal wins when several fire together?** The independent slots have no shared cash, so a real account could not take every signal in a sell-off. That rule is a spec decision.
- **Parameter sensitivity.** Only the script's setting and `max_units` are tried. A grid over the channel lengths would need the optimizer to learn the Turtle strategy first, with the usual out-of-sample discipline.

Write what you conclude from the tables above here, in your own words, before relying on any of them.